# 4. Is the difference real?

**Going further.** Assumes you have run a regression and met a p-value.

The previous notebooks answered questions about data that already existed.
This one is about data you are about to create: you change something, measure
two groups, and one number is higher. The whole question is whether that means
anything.

Three things, in the order they actually happen:

1. **Before you start**, how many observations do you need for the test to be
   capable of answering?
2. **When you assign**, how do you split without biasing it?
3. **When it ends**, is the difference bigger than the noise?

**Step 1 is the one people skip**, and skipping it is why so many experiments
end in a shrug. A test too small to detect the effect you care about produces
"no significant difference" whether or not there is one, and those two
outcomes are indistinguishable afterwards.

## 1. How big does this need to be?

Say you run a shop. Average order is \$100, orders vary with a standard
deviation of \$50, and you would change the checkout page if it lifted the
average by 5%.

That last clause is the important one. **"Any improvement" is not a target.**
You must name the smallest effect worth acting on, because that is what sets
the size.

In [ ]:
from otter import sample_size_for_mean

size = sample_size_for_mean(
    baseline_mean=100.0,
    baseline_sd=50.0,
    min_lift_pct=5.0,     # a 5% relative lift, how targets are actually stated
    power=0.80,           # an 80% chance of detecting it if it is really there
    alpha=0.05,
)
size

**`power=0.80` deserves a sentence.** It means that if the effect is genuinely
there at the size you named, you have an 80% chance of seeing it. One run in
five misses a real effect. That is the *conventional* number, not a safe one,
and it is worth knowing you accepted it.

Now watch what chasing a smaller effect costs.

In [ ]:
for lift in (10.0, 5.0, 2.0, 1.0):
    s = sample_size_for_mean(100.0, 50.0, lift)
    print(f"{lift:>5.1f}% lift  ->  {s.total_n:>9,} observations")

**Halving the effect roughly quadruples the sample.** This is the single most
useful fact in experiment design and it is not intuitive.

It is also where the honest conversation happens: if detecting a 1% lift needs
more traffic than you get in a year, you cannot answer that question, and
finding out now is a good day. Finding out after six weeks of running is not.

## 2. Assign without fooling yourself

Splitting by something convenient, first letter, sign-up date, whichever server
took them, hides a systematic difference inside your two groups. Random
assignment is what makes the arms comparable.

`seed` makes the split reproducible, which is what lets somebody else check
your work rather than take your word for it.

In [ ]:
import numpy as np
import pandas as pd
from otter import assign_groups

rng = np.random.default_rng(7)
n = 4000

customers = pd.DataFrame({
    "customer_id": range(n),
    # Spend BEFORE the experiment. Keep this: section 3 uses it.
    "spend_before": rng.gamma(shape=4.0, scale=25.0, size=n),
})

split = assign_groups(customers["customer_id"], {"control": 0.5, "treatment": 0.5}, seed=7)
customers = customers.merge(split, left_on="customer_id", right_on="unit")
customers["group"].value_counts()

### Check the split before trusting it

Randomisation works on average, not in every instance. Check that the arms
actually look alike on what you knew beforehand. If they do not, you learn it
now rather than mistaking a pre-existing gap for your effect.

In [ ]:
from otter import pretest_balance

pretest_balance(customers, group_col="group", metric_cols=["spend_before"])

## 3. Run it, and read the result

Simulate an experiment where the treatment genuinely adds about 4%. Because we
built it, we know the truth, which is the only way to see whether the method
finds it.

In [ ]:
TRUE_LIFT = 0.04

# After-spend is mostly the same customer as before, plus noise, plus the
# treatment effect for the treated arm. That persistence is realistic and it
# is what CUPED exploits below.
noise = rng.normal(0, 18, size=len(customers))
customers["spend_after"] = customers["spend_before"] + noise
treated = customers["group"].eq("treatment")
customers.loc[treated, "spend_after"] *= (1 + TRUE_LIFT)

from otter import compare_groups

compare_groups(
    customers,
    group_col="group",
    metric_cols=["spend_after"],
    control="control",
    treatment="treatment",
)

**Read the confidence interval before the p-value.** The interval says which
effect sizes are consistent with what you saw. "Somewhere between +1% and +7%"
is a usable finding; "p < 0.05" on its own tells you almost nothing about
whether to act.

A wide interval that excludes zero means: something is there, and you do not
yet know how big. That is a real and common result, and saying so is better
than rounding it to "it worked".

## Getting more out of the same data

Most of the variation in after-spend is just *who the customer already was*.
That variation is noise as far as your question goes, and you measured it
before the experiment started.

CUPED subtracts the predictable part, shrinking the noise without touching the
effect. Same customers, same data, tighter answer.

In [ ]:
adjusted = compare_groups(
    customers,
    group_col="group",
    metric_cols=["spend_after"],
    control="control",
    treatment="treatment",
    cuped_pre={"spend_after": "spend_before"},
)
adjusted

Compare the interval width against the previous cell. It should be narrower,
and the estimated effect should sit in much the same place.

**That is the tell for a legitimate variance reduction: the precision improves
and the estimate does not move.** If the estimate jumps, something is wrong,
most often a pre-period variable that was not actually measured before
treatment began. A covariate contaminated by the treatment will happily produce
a tighter interval around a wrong number.

## The honest closing note

We knew the true lift was 4% because we wrote it. Real experiments do not come
with an answer key, which is why the discipline matters:

- Decide the smallest effect worth acting on **before** you look
- Size it, and say so out loud if the answer is "not enough data"
- Randomise, then check the randomisation held
- Report the interval, not just whether a threshold was crossed

**Try this:** set `TRUE_LIFT = 0.0` and run it again. Roughly one run in twenty
will show a "significant" result anyway. That is not a bug, it is what
`alpha=0.05` means, and seeing it once is worth more than reading it ten times.